In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

try:
    import kagglehub   # only available on Kaggle
except ImportError:
    pass
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.model_selection import KFold
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

DATA_DIR = "/kaggle/input/competitions/domestic-flight-price-prediction-challenge"
if not os.path.isdir(DATA_DIR):
    DATA_DIR = "."   # running from the repository root

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
sample = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

print("Train:", train.shape)
print("Test:", test.shape)
print("Sample submission:", sample.shape)

display(train.head())
display(test.head())
display(sample.head())

In [ ]:
print("Training columns:")
print(train.columns.tolist())

print("\nTest columns:")
print(test.columns.tolist())

print("\nMissing values in training data:")
display(train.isna().sum().sort_values(ascending=False))

print("\nMissing values in test data:")
display(test.isna().sum().sort_values(ascending=False))

assert "price" in train.columns
assert "price" not in test.columns
assert "id" in test.columns
assert list(sample.columns) == ["id", "price"]

In [ ]:
# Preserve the original target and IDs
y = train["price"].astype(float).copy()
train_ids = train["id"].copy()
test_ids = test["id"].copy()

# Combine features only; never include price in the combined data
train_features = train.drop(columns=["price", "id"]).copy()
test_features = test.drop(columns=["id"]).copy()

n_train = len(train_features)
full = pd.concat(
    [train_features, test_features],
    ignore_index=True
)

# Extract airline code and flight number
flight_parts = full["flight"].astype("string").str.extract(
    r"^\s*(\S+)\s+(\d+)\s*$"
)

full["airline_code"] = flight_parts[0].fillna("UNKNOWN")
full["flight_num"] = pd.to_numeric(
    flight_parts[1], errors="coerce"
)

# Recover missing airline using the observed airline-code mapping
airline_map = (
    full.dropna(subset=["airline"])
        .groupby("airline_code")["airline"]
        .agg(lambda s: s.mode().iloc[0])
        .to_dict()
)

full["airline"] = full["airline"].fillna(
    full["airline_code"].map(airline_map)
)

full["airline"] = full["airline"].fillna("Unknown")

# Create route feature
full["route"] = (
    full["source_city"].astype(str)
    + ">"
    + full["destination_city"].astype(str)
)

# Convert duration to numeric if necessary
full["duration"] = pd.to_numeric(
    full["duration"], errors="coerce"
)

# Impute missing duration from progressively broader groups
for cols in [
    ["flight", "route", "stops"],
    ["route", "stops", "departure_time"],
    ["route", "stops"],
    ["route"]
]:
    group_median = full.groupby(
        cols, dropna=False
    )["duration"].transform("median")

    full["duration"] = full["duration"].fillna(group_median)

full["duration"] = full["duration"].fillna(
    full["duration"].median()
)

# Encode stops as an ordinal feature
stop_map = {
    "zero": 0,
    "one": 1,
    "two_or_more": 2
}

full["stops_n"] = full["stops"].map(stop_map).fillna(-1)

# Time categories
time_map = {
    "Early_Morning": 0,
    "Morning": 1,
    "Afternoon": 2,
    "Evening": 3,
    "Night": 4,
    "Late_Night": 5
}

full["dep_t"] = full["departure_time"].map(time_map).fillna(-1)
full["arr_t"] = full["arrival_time"].map(time_map).fillna(-1)

# Business-class indicator and interaction
full["is_biz"] = (full["flight_class"] == "Business").astype(int)
full["biz_x_dur"] = full["is_biz"] * full["duration"]

# Days remaining features
full["days_left"] = pd.to_numeric(
    full["days_left"], errors="coerce"
)

full["log_days"] = np.log1p(
    full["days_left"].clip(lower=0)
)

full["days_bucket"] = pd.cut(
    full["days_left"],
    bins=[-1, 1, 2, 7, 14, 21, 30, 60, np.inf],
    labels=False
).fillna(-1).astype(int)

# Useful categorical interactions
full["airline_class"] = (
    full["airline"].astype(str)
    + "_"
    + full["flight_class"].astype(str)
)

full["route_class"] = (
    full["route"].astype(str)
    + "_"
    + full["flight_class"].astype(str)
)

# Remove the original flight string; extracted features are retained
full = full.drop(columns=["flight"])

# Split back into train and test
X_train = full.iloc[:n_train].copy().reset_index(drop=True)
X_test = full.iloc[n_train:].copy().reset_index(drop=True)

# Categorical columns for LightGBM
cat_cols = [
    "airline",
    "airline_code",
    "source_city",
    "destination_city",
    "route",
    "departure_time",
    "arrival_time",
    "stops",
    "flight_class",
    "airline_class",
    "route_class"
]

# Use identical category definitions in train and test
for col in cat_cols:
    categories = full[col].astype("string").fillna("Unknown").unique()

    X_train[col] = pd.Categorical(
        X_train[col].astype("string").fillna("Unknown"),
        categories=categories
    )

    X_test[col] = pd.Categorical(
        X_test[col].astype("string").fillna("Unknown"),
        categories=categories
    )

# Check the final matrices
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Missing duration:", X_train["duration"].isna().sum(),
      X_test["duration"].isna().sum())

display(X_train.head())

In [ ]:
def build_model():
    return lgb.LGBMRegressor(
        objective="regression",
        n_estimators=1200,
        learning_rate=0.03,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=15,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

print("LightGBM model is ready.")

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_predictions = {
    "raw": np.zeros(len(y)),
    "log": np.zeros(len(y))
}

for fold, (train_idx, valid_idx) in enumerate(kf.split(X_train), start=1):
    print(f"\n========== Fold {fold}/5 ==========")

    for target_name in ["raw", "log"]:
        model = build_model()

        y_fit = (
            np.log(y.iloc[train_idx])
            if target_name == "log"
            else y.iloc[train_idx]
        )

        model.fit(
            X_train.iloc[train_idx],
            y_fit,
            categorical_feature=cat_cols
        )

        pred = model.predict(X_train.iloc[valid_idx])

        if target_name == "log":
            pred = np.exp(pred)

        pred = np.clip(pred, 0, None)
        oof_predictions[target_name][valid_idx] = pred

        rmse = np.sqrt(
            mean_squared_error(y.iloc[valid_idx], pred)
        )

        print(f"{target_name.upper()} price RMSE: {rmse:.2f}")

# Compare full out-of-fold predictions
results = []

for name, pred in oof_predictions.items():
    results.append({
        "target": name,
        "R2": r2_score(y, pred),
        "RMSE": np.sqrt(mean_squared_error(y, pred)),
        "MAE": mean_absolute_error(y, pred),
        "MAPE_percent": (
            np.mean(np.abs(pred - y.to_numpy()) /
                    np.maximum(np.abs(y.to_numpy()), 1e-8)) * 100
        )
    })

results_df = pd.DataFrame(results).sort_values("RMSE")

display(results_df.round(4))

best_target = results_df.iloc[0]["target"]
print("Best target transformation:", best_target)

In [ ]:
final_model = build_model()

y_final = np.log(y) if best_target == "log" else y

final_model.fit(
    X_train,
    y_final,
    categorical_feature=cat_cols
)

test_predictions = final_model.predict(X_test)

if best_target == "log":
    test_predictions = np.exp(test_predictions)

test_predictions = np.clip(test_predictions, 0, None)

print("Predictions generated:", len(test_predictions))
print("Minimum predicted price:", test_predictions.min())
print("Maximum predicted price:", test_predictions.max())
print("Missing predictions:", np.isnan(test_predictions).sum())

In [ ]:
prediction_map = pd.Series(
    test_predictions,
    index=test_ids
)

submission = sample[["id"]].copy()
submission["price"] = submission["id"].map(prediction_map)

assert len(submission) == len(sample)
assert submission["id"].equals(sample["id"])
assert submission["price"].notna().all()
assert np.isfinite(submission["price"]).all()

OUTPUT_PATH = "/kaggle/working/submission.csv" if os.path.isdir("/kaggle/working") else "submission.csv"
submission.to_csv(OUTPUT_PATH, index=False)

print("Submission saved to:", OUTPUT_PATH)
print("Submission shape:", submission.shape)
display(submission.head(10))

In [ ]:
importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": final_model.feature_importances_
}).sort_values("importance", ascending=False)

display(importance.head(20))

import matplotlib.pyplot as plt

top_features = importance.head(20).sort_values("importance")

plt.figure(figsize=(10, 7))
plt.barh(top_features["feature"], top_features["importance"])
plt.xlabel("Feature importance")
plt.title("Top 20 LightGBM Features")
plt.tight_layout()
plt.show()